In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from strong_heuristic_local_executor import (
    OFFSET_SET_DB,
    SAFE_EXTENDED_OFFSET_SET_DB,
    EXTENDED_1DB_OFFSET_SET_DB,
    _snap_to_set,
    strong_directional_heuristic_local_executor,
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1 — Offset sets

In [ ]:
print('OFFSET_SET_DB            :', OFFSET_SET_DB)
print('SAFE_EXTENDED_OFFSET_SET :', SAFE_EXTENDED_OFFSET_SET_DB)
print('EXTENDED_1DB_OFFSET_SET  :', EXTENDED_1DB_OFFSET_SET_DB)

## 2 — Proportional bias → offset mapping (no safety)

The mapping is **asymmetric**:
- Negative side: `bias × 12`  → full range down to −12 dB
- Positive side: `bias × 6`   → up to +6 dB

After computing the continuous `proto_offset` the result is **snapped to the nearest 1 dB step**.

In [ ]:
bias_vals = np.linspace(-1.0, 1.0, 500)

def raw_offset(b):
    if abs(b) < 1e-9:
        return 0.0
    return b * 12.0 if b < 0 else b * 6.0

raw  = np.array([raw_offset(b) for b in bias_vals])
snapped = np.array([_snap_to_set(raw_offset(b), EXTENDED_1DB_OFFSET_SET_DB) for b in bias_vals])

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(bias_vals, raw, lw=2, label='proto_offset (continuous)')
ax.step(bias_vals, snapped, where='mid', lw=1.5, linestyle='--', label='applied_offset (1 dB snap)')
ax.axhline(0, color='k', lw=0.6)
ax.axvline(0, color='k', lw=0.6)
ax.set_xlabel('bias  b')
ax.set_ylabel('offset  (dB)')
ax.set_title('Proportional bias → A3 offset (no safety, no load)')
ax.yaxis.set_major_locator(ticker.MultipleLocator(2))
ax.set_yticks(np.arange(-12, 7, 2))
ax.legend()
plt.tight_layout()
plt.show()

# Key reference points
for b in [-1.0, -0.5, -0.1, -0.01, 0.0, 0.1, 0.5, 1.0]:
    ro = raw_offset(b)
    so = _snap_to_set(ro, EXTENDED_1DB_OFFSET_SET_DB)
    print(f'  bias={b:+.3f}  raw={ro:+6.2f} dB  snapped={so:+.0f} dB')

## 3 — Effect of load safety

`load_safety = alpha_load × max(target_load − l_safe, 0)`

This term is always ≥ 0 and **pushes the offset toward zero** (less aggressive offloading).

In [ ]:
alpha_load = 12.0
l_safe = 0.80

bias_grid   = np.linspace(-1.0, 0.0, 200)
load_levels = [0.60, 0.75, 0.80, 0.85, 0.90, 1.00]

fig, ax = plt.subplots(figsize=(9, 4))
for tl in load_levels:
    load_safety = alpha_load * max(tl - l_safe, 0.0)
    proto = np.clip(np.array([raw_offset(b) for b in bias_grid]) + load_safety, -12, 6)
    snapped_line = np.array([_snap_to_set(v, EXTENDED_1DB_OFFSET_SET_DB) for v in proto])
    ax.step(bias_grid, snapped_line, where='mid', label=f'target_load={tl:.2f}')

ax.axhline(0, color='k', lw=0.6)
ax.set_xlabel('bias  b')
ax.set_ylabel('applied offset  (dB)')
ax.set_title('A3 offset vs bias for different target loads  (alpha_load=12)')
ax.yaxis.set_major_locator(ticker.MultipleLocator(2))
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 4 — Effect of mobility safety

`mobility_safety = alpha_mobility × (HF_ratio + PP_ratio)`

High handover-failure or ping-pong rates reduce aggressiveness in the same direction as load safety.

In [ ]:
alpha_mobility = 3.0

mob_levels = [0.0, 0.2, 0.4, 0.6, 1.0]

fig, ax = plt.subplots(figsize=(9, 4))
for mob in mob_levels:
    mob_safety = alpha_mobility * mob
    proto = np.clip(np.array([raw_offset(b) for b in bias_grid]) + mob_safety, -12, 6)
    snapped_line = np.array([_snap_to_set(v, EXTENDED_1DB_OFFSET_SET_DB) for v in proto])
    ax.step(bias_grid, snapped_line, where='mid', label=f'HF+PP={mob:.1f}')

ax.axhline(0, color='k', lw=0.6)
ax.set_xlabel('bias  b')
ax.set_ylabel('applied offset  (dB)')
ax.set_title('A3 offset vs bias for different mobility risk  (alpha_mobility=3)')
ax.yaxis.set_major_locator(ticker.MultipleLocator(2))
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 5 — Heatmap: bias × target_load → applied offset

In [ ]:
bias_ax  = np.linspace(-1.0, 0.0, 60)
load_ax  = np.linspace(0.5, 1.0, 60)

Z = np.zeros((len(load_ax), len(bias_ax)))
for i, tl in enumerate(load_ax):
    ls = alpha_load * max(tl - l_safe, 0.0)
    for j, b in enumerate(bias_ax):
        proto = float(np.clip(raw_offset(b) + ls, -12, 6))
        Z[i, j] = _snap_to_set(proto, EXTENDED_1DB_OFFSET_SET_DB)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.contourf(bias_ax, load_ax, Z, levels=np.arange(-13, 7, 1), cmap='RdYlGn')
cb = plt.colorbar(im, ax=ax)
cb.set_label('applied offset (dB)')
ax.axhline(l_safe, color='white', lw=1.5, linestyle='--', label=f'l_safe={l_safe}')
ax.set_xlabel('bias  b')
ax.set_ylabel('target load')
ax.set_title('Applied A3 offset (dB) — bias × target_load  (no mobility risk)')
ax.legend()
plt.tight_layout()
plt.show()

## 6 — Toy scenario: 3 gNBs, 2 slices, 6 UEs

Topology: gNB-0 ↔ gNB-1 ↔ gNB-2 (linear).

Upper PPO sends a mixed bias matrix:
- gNB-0 eMBB: slight offload (`b = -0.15`)
- gNB-1 eMBB: heavy offload (`b = -0.80`)
- gNB-2 eMBB: retain (`b = +0.40`)
- All URLLC: neutral

In [ ]:
rng = np.random.default_rng(42)

NUM_GNB   = 3
NUM_SLICE = 2   # eMBB=0, URLLC=1
NUM_UE    = 6
SLICE_TYPES = ('eMBB', 'URLLC')

neighbor_graph = {0: [1], 1: [0, 2], 2: [1]}
max_neighbors  = max(len(v) for v in neighbor_graph.values())  # 2

# Bias matrix  [gnb, slice]
B = np.array([
    [-0.15,  0.00],   # gNB-0: slight offload eMBB, neutral URLLC
    [-0.80,  0.00],   # gNB-1: heavy offload eMBB
    [ 0.40,  0.00],   # gNB-2: retain eMBB
])

# UEs: 2 per gNB, all eMBB
ue_serving_gnb = np.array([0, 0, 1, 1, 2, 2])
ue_slice       = np.zeros(NUM_UE, dtype=int)   # all eMBB

# RSRP matrix  [ue, gnb]  (dBm)
rsrp_matrix = np.array([
    [-70, -80, -95],  # UE-0 @ gNB-0
    [-72, -82, -97],  # UE-1 @ gNB-0
    [-82, -68, -78],  # UE-2 @ gNB-1
    [-84, -70, -80],  # UE-3 @ gNB-1
    [-96, -79, -65],  # UE-4 @ gNB-2
    [-98, -81, -67],  # UE-5 @ gNB-2
], dtype=float)

load          = np.array([[0.55, 0.30], [0.75, 0.40], [0.60, 0.25]])  # [gnb, slice]
sla_violation = np.zeros((NUM_GNB, NUM_SLICE))
ho_failure_ratio = np.zeros((NUM_GNB, max_neighbors, NUM_SLICE))
pingpong_ratio   = np.zeros((NUM_GNB, max_neighbors, NUM_SLICE))
prev_offsets     = np.zeros((NUM_GNB, max_neighbors, NUM_SLICE))

print('Load matrix (gnb × slice):')
print(load)
print('\nBias matrix (gnb × slice):')
print(B)

In [ ]:
offsets, dbg = strong_directional_heuristic_local_executor(
    B=B,
    prev_offsets=prev_offsets,
    ue_slice=ue_slice,
    ue_serving_gnb=ue_serving_gnb,
    rsrp_matrix=rsrp_matrix,
    neighbor_graph=neighbor_graph,
    load=load,
    sla_violation=sla_violation,
    ho_failure_ratio=ho_failure_ratio,
    pingpong_ratio=pingpong_ratio,
    return_debug=True,
)

print(f'offsets shape: {offsets.shape}  (gnb × neighbor_slot × slice)\n')

for (gnb, nbr, sl), d in sorted(dbg.items()):
    slice_name = SLICE_TYPES[sl]
    print(
        f'  gNB-{gnb} → gNB-{nbr}  [{slice_name}]  '
        f'bias={d["bias"]:+.3f}  '
        f'raw={d["raw_offset_db"]:+6.2f} dB  '
        f'load_safety={d["load_safety_db"]:+5.2f}  '
        f'mob_safety={d["mobility_safety_db"]:+5.2f}  '
        f'→ {d["applied_offset_db"]:+.0f} dB  '
        f'[safe={d["target_is_safe"]} radio={d["radio_feasible"]}]'
    )

In [ ]:
import matplotlib.patches as mpatches

records = [
    (gnb, nbr, sl, d)
    for (gnb, nbr, sl), d in sorted(dbg.items())
    if sl == 0   # eMBB only
]

labels = [f'gNB-{g}→{n}' for g, n, _, _ in records]
raw_vals     = [d['raw_offset_db']       for _, _, _, d in records]
applied_vals = [d['applied_offset_db']   for _, _, _, d in records]
safety_vals  = [d['load_safety_db'] + d['mobility_safety_db'] for _, _, _, d in records]

x = np.arange(len(labels))
w = 0.28

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - w, raw_vals,     w, label='raw offset (proportional)', color='steelblue')
ax.bar(x,     safety_vals,  w, label='safety correction (+)',     color='tomato')
ax.bar(x + w, applied_vals, w, label='applied offset (snapped)',  color='seagreen')

ax.axhline(0, color='k', lw=0.7)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('offset (dB)')
ax.set_title('Toy scenario — eMBB offsets per direction')
ax.yaxis.set_major_locator(ticker.MultipleLocator(2))
ax.legend()
plt.tight_layout()
plt.show()

## 7 — Sensitivity to alpha_load and alpha_mobility

For a fixed negative bias (`b = -0.5`, raw = −6 dB), how much does each safety parameter pull the offset back toward zero?

In [ ]:
b_fixed    = -0.5
raw_fixed  = raw_offset(b_fixed)   # -6.0 dB
load_over  = 0.10                  # target is 10 pp above l_safe
mob_risk   = 0.30                  # HF+PP = 0.30

alpha_load_range = np.linspace(0, 24, 100)
alpha_mob_range  = np.linspace(0, 10, 100)

applied_load = np.array([
    _snap_to_set(float(np.clip(raw_fixed + a * load_over, -12, 6)), EXTENDED_1DB_OFFSET_SET_DB)
    for a in alpha_load_range
])
applied_mob = np.array([
    _snap_to_set(float(np.clip(raw_fixed + a * mob_risk, -12, 6)), EXTENDED_1DB_OFFSET_SET_DB)
    for a in alpha_mob_range
])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.step(alpha_load_range, applied_load, where='mid', color='steelblue')
ax1.axvline(12, color='k', linestyle='--', lw=0.8, label='default alpha_load=12')
ax1.set_xlabel('alpha_load')
ax1.set_ylabel('applied offset (dB)')
ax1.set_title(f'bias={b_fixed}, load_over={load_over}')
ax1.yaxis.set_major_locator(ticker.MultipleLocator(2))
ax1.legend()

ax2.step(alpha_mob_range, applied_mob, where='mid', color='tomato')
ax2.axvline(3, color='k', linestyle='--', lw=0.8, label='default alpha_mobility=3')
ax2.set_xlabel('alpha_mobility')
ax2.set_ylabel('applied offset (dB)')
ax2.set_title(f'bias={b_fixed}, HF+PP={mob_risk}')
ax2.yaxis.set_major_locator(ticker.MultipleLocator(2))
ax2.legend()

plt.suptitle('Sensitivity of applied offset to safety hyperparameters')
plt.tight_layout()
plt.show()

## 8 — Bias sweep over time (simulated)

Simulate a single (gNB-0 → gNB-1, eMBB) direction over 30 steps where the upper PPO ramps from neutral to heavy offload and back, with EMA smoothing enabled.

In [ ]:
steps = 30
t     = np.arange(steps)
bias_seq = np.concatenate([
    np.linspace(0, -1.0, 10),
    np.linspace(-1.0, 0, 10),
    np.linspace(0, 0.6, 10),
])

ETA = 0.4
applied_no_smooth = []
applied_smooth    = []
prev_no = 0.0
prev_sm = 0.0

for b in bias_seq:
    ro  = raw_offset(b)
    proto = float(np.clip(ro, -12, 6))

    snapped = _snap_to_set(proto, EXTENDED_1DB_OFFSET_SET_DB)
    applied_no_smooth.append(snapped)
    prev_no = snapped

    smoothed = (1 - ETA) * proto + ETA * prev_sm
    snapped_sm = _snap_to_set(float(np.clip(smoothed, -12, 6)), EXTENDED_1DB_OFFSET_SET_DB)
    applied_smooth.append(snapped_sm)
    prev_sm = snapped_sm

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax1.plot(t, bias_seq, color='purple', lw=1.5)
ax1.axhline(0, color='k', lw=0.5)
ax1.set_ylabel('bias  b')
ax1.set_title('Upper PPO bias over time')

ax2.step(t, applied_no_smooth, where='mid', label='no smoothing (eta=0)', lw=1.5)
ax2.step(t, applied_smooth,    where='mid', label=f'EMA smoothing (eta={ETA})', lw=1.5, linestyle='--')
ax2.axhline(0, color='k', lw=0.5)
ax2.set_xlabel('step')
ax2.set_ylabel('applied A3 offset (dB)')
ax2.set_title('Applied offset — proportional mapping + 1 dB snap')
ax2.yaxis.set_major_locator(ticker.MultipleLocator(2))
ax2.legend()
plt.tight_layout()
plt.show()